# Coco VM — a Colab GPU driven from a machine of yours

Three cells. The first opens an SSH tunnel into this runtime, the second keeps
the runtime alive, the third is optional and builds the stack here. The full
notebook, `cocolog_colab.ipynb`, is the one that trains and serves from cells;
this one exists so that another machine — a laptop running Claude Code, say —
can do the build, the training and the measuring over SSH instead.

**Before the first cell, one secret.** The tunnel authorises whatever public key
the Colab secret `SSH_PUBKEY` holds: the key icon in the left bar, name
`SSH_PUBKEY`, value the *public* half of the driving machine's key (the line in
its `~/.ssh/<key>.pub`), notebook access on. No key lives in this notebook.
Without the secret the cell falls back to the keys GitHub publishes for
`GITHUB_USER`.

**Runtime → Change runtime type → GPU** first. And Colab's usage rules list SSH
among the things disallowed on its runtimes: the usual cost is a killed runtime,
and it is your account and your call.


## 1 · The tunnel

Installs `sshd` for public keys only, passwords off, on a config of its own —
Colab's stock `sshd_config` says port 2222 on loopback and that port is Colab's
already — and dials out through a Cloudflare quick tunnel, so nothing listens on
the internet. It prints the hostname; that is what the driving machine needs:

    ssh -o ProxyCommand="cloudflared access ssh --hostname %h" root@<host>.trycloudflare.com

The tunnel lives as long as this runtime does. Rerunning the cell replaces it,
and the hostname changes.


In [ ]:
# == SSH into this Colab VM through a Cloudflare quick tunnel ==
# Root login by public key ONLY, passwords off. NO KEY LIVES IN THIS
# NOTEBOOK: it comes from the Colab secret SSH_PUBKEY (the key icon in the
# left bar: paste a public key, allow notebook access) or, failing that,
# from the public keys GITHUB_USER has registered on GitHub.
import subprocess, os, re, time, urllib.request
GITHUB_USER = 'saman-pasha'   # the fallback; '' to insist on the secret
keys = ''
try:
    from google.colab import userdata
    keys = userdata.get('SSH_PUBKEY').strip()
    print('authorised: the Colab secret SSH_PUBKEY')
except Exception:
    pass
if not keys and GITHUB_USER:
    try:
        keys = urllib.request.urlopen(f'https://github.com/{GITHUB_USER}.keys', timeout=20).read().decode().strip()
        print(f'authorised: the {len(keys.splitlines())} public key(s) GitHub publishes for {GITHUB_USER}')
    except Exception as e:
        print('GitHub keys unavailable:', e)
if not keys:
    raise SystemExit('no public key to authorise -- add the Colab secret SSH_PUBKEY, or set GITHUB_USER')
subprocess.run('DEBIAN_FRONTEND=noninteractive apt-get -qq update >/dev/null && '
               'DEBIAN_FRONTEND=noninteractive apt-get -qq install -y openssh-server >/dev/null',
               shell=True, check=True)
os.makedirs('/root/.ssh', mode=0o700, exist_ok=True)
open('/root/.ssh/authorized_keys', 'w').write(keys + '\n'); os.chmod('/root/.ssh/authorized_keys', 0o600)
open('/root/.ssh/environment', 'w').write(
    'CICILI=/content/cicili\nZIGURATIP=/content/ZiguratIP\nZIGURATIP_HOME=/content/ZiguratIP/home\n'
    'COCOLOG=/content/cocolog\nLD_LIBRARY_PATH=/content/ZiguratIP/home/lib\n'
    'PATH=/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin\n')
# Colab's own /etc/ssh/sshd_config says `Port 2222' on loopback, and that
# port is Colab's already -- so sshd gets a config of its own: 22, loopback.
CONF = '/content/sshd_config'
open(CONF, 'w').write(
    'Port 22\nListenAddress 127.0.0.1\n'
    'HostKey /etc/ssh/ssh_host_ed25519_key\nHostKey /etc/ssh/ssh_host_rsa_key\n'
    'PermitRootLogin prohibit-password\nPubkeyAuthentication yes\nAuthorizedKeysFile .ssh/authorized_keys\n'
    'PasswordAuthentication no\nKbdInteractiveAuthentication no\nUsePAM no\n'
    'PermitUserEnvironment yes\nClientAliveInterval 30\nSubsystem sftp /usr/lib/openssh/sftp-server\n')
os.makedirs('/run/sshd', exist_ok=True); subprocess.run(['ssh-keygen', '-A'], check=True)
old = globals().get('_sshd')
if old is not None and old.poll() is None: old.terminate()
globals()['_sshd'] = subprocess.Popen(['/usr/sbin/sshd', '-D', '-e', '-f', CONF],
                                      stdout=open('/content/sshd.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(2)
if globals()['_sshd'].poll() is not None:
    raise SystemExit('sshd did not start:\n' + open('/content/sshd.log').read()[-1500:])
CF = '/content/cloudflared'
if not os.path.exists(CF):
    subprocess.run(['curl', '-sSL', '-o', CF, 'https://github.com/cloudflare/cloudflared/releases/latest/'
                    'download/cloudflared-linux-amd64'], check=True); os.chmod(CF, 0o755)
old = globals().get('_ssh_tunnel')
if old is not None and old.poll() is None: old.terminate()
LOG = '/content/cloudflared-ssh.log'
_ssh_tunnel = subprocess.Popen([CF, 'tunnel', '--url', 'ssh://127.0.0.1:22', '--no-autoupdate'],
                               stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
globals()['_ssh_tunnel'] = _ssh_tunnel
host = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://([a-z0-9-]+\.trycloudflare\.com)', open(LOG).read())
    if m: host = m.group(1); break
    if _ssh_tunnel.poll() is not None: break
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none visible -- Runtime > Change runtime type > GPU, then rerun')
if host:
    print(f'\nfrom the authorised machine:\n\n    ssh -o ProxyCommand="cloudflared access ssh --hostname %h" root@{host}\n')
    print('the tunnel lives as long as this runtime does; leave the notebook open')
else:
    print('no tunnel URL after 60 s -- the log:'); print(open(LOG).read()[-2000:])

## 2 · Keep the runtime alive

Colab reclaims a runtime it thinks idle, and a runtime with an idle GPU gets a
notice first. This cell touches the GPU once a minute for up to six hours. It
blocks its own cell only; the tunnel keeps running underneath. Leave it running
and leave the tab open.


In [ ]:
# keep the runtime alive: a GPU touch every minute, for up to six hours
import time, torch
t0 = time.time()
while time.time() - t0 < 6 * 3600:
    x = torch.randn(2048, 2048, device='cuda')
    y = (x @ x).sum().item()
    print(time.strftime('%H:%M:%S'), 'alive, gpu touch', round(y, 1), flush=True)
    time.sleep(60)


## 3 · (Optional) build the stack here, with the repository's own installer

The driving machine can do this over the tunnel — `install/install-linux.sh` is
what it runs — so this cell is for a VM that should build itself. It clones the
three repositories beside each other under `/content`, installs the packages and
a clang 16+ (clang 18 from apt.llvm.org on this Ubuntu), builds Cicili's Lisp
side, ZiguratIP, cocolog, the Parsi objects and every loadable module whose
dependency is present — `library(torch)` from the pip torch this runtime already
has, CUDA included. Ten minutes or so on Colab's two CPUs. Run it after cell 1,
in its own cell, so the keep-alive does not block it.


In [ ]:
import os, subprocess
for repo in ('cicili', 'ZiguratIP', 'cocolog'):
    path = f'/content/{repo}'
    if not os.path.isdir(path):
        subprocess.run(['git', 'clone', '-q', f'https://github.com/saman-pasha/{repo}.git', path], check=True)
    else:
        subprocess.run(['git', '-C', path, 'pull', '-q', '--ff-only'], check=True)
    print(repo, subprocess.run(['git', '-C', path, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
env = dict(os.environ, CICILI='/content/cicili', ZIGURATIP='/content/ZiguratIP',
           ZIGURATIP_HOME='/content/ZiguratIP/home', COCOLOG='/content/cocolog')
proc = subprocess.Popen(['sh', 'install/install-linux.sh'], cwd='/content/cocolog', env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('install exit', proc.wait())
print(subprocess.run(['ls', '-la', '/content/cocolog/cocolog', '/content/cocolog/library/torch.so'],
                     capture_output=True, text=True).stdout)
